# 1-B: Agent, where the AI decides the route; tools and MCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/en/notebooks/1-B_agent.ipynb)

> 📍 **Matching slides**: Session 1, "Stop 3: Agent 1-B", slides **30–39**; the core-loop exercise is appendix slide **47** (class time 1:10–1:40)

## What this notebook does

In the helper from 1-A, the steps, branches and stopping conditions were all arranged by the program (a workflow). This notebook takes a different approach: **no flow diagram, just a toolbox for the AI**.
You tell it which tools exist and what the goal is, and let it decide which to use first and how many times. This approach is called an **agent**:

> **Agent = model + tools + loop**: ask the model -> it says which tool it wants, and your program runs it and pastes the result back -> ask the model again -> until it says "no more tools".
> The model itself does not run code; it only "says" which tool it wants, and your program does the real work (details in 1-0).

The toolbox has three tools we wrote ourselves (Part 6 also connects two read-only tools from an MCP server):

| Tool | What it does |
| --- | --- |
| `run_code` | Runs the homework under teaching limits and returns the output or the error |
| `check_indent` | Checks indentation and the length of each line (deliberately does not check naming, which is left for Session 2) |
| `read_spec` | Reads the assignment requirements for this problem |

| Part | What you will do | Matching slides |
| --- | --- | --- |
| Part 1 | Prepare the three tools, and test run_code's limits and error messages | 31–32 |
| Part 2 | Look at the tool descriptions the model receives | 33 |
| Part 3 | Build a local agent and watch its tool-call record | 34, 35 |
| Part 4 | Run three times and compare, then adjust the system prompt | 36 |
| Part 5 | Tell apart stopping by itself, hitting the step limit, and success | 37–38 |
| Part 6 (instructor demo) | Connect MCP tools; the tool-description length comparison is optional reading | 39 |
| Part 7 (optional after class) | Rebuild the core loop with LangGraph | 47 |

## How to use it

1. First run **"0. Environment setup"**.
2. The only things you change are form fields: **homework**, **spec** (the assignment requirements, in Part 3), and **system prompt extra** (the sentence added to the system prompt, in Part 4).
3. Every part: **🤔 write your prediction first → ▶ run → 👀 compare**. For the core, do Parts 1–5 in order; follow the instructor for Part 6, and Part 7 is optional after class.
4. If the runtime has been reset (not just a browser disconnect), rerun "0. Environment setup" first. It also writes the tools and `agent.py` back; then return to the part you want. The file-writing cells only need to be run, not read line by line.

## 🎯 The mini project for these two sessions: the Homework Helper

The **Homework Helper** is a small program that helps check Python homework. You give it a piece of homework, and it replies with the review result.
In the workflow version (1-A, workflow), a syntax error gets an error reply first; only code with valid syntax goes on to the AI, which writes feedback containing **"Strengths:" and "Improvements:"**.
If the format check keeps failing, the flow stops when it reaches the maximum number of generations and marks the result as not passed. "Stopped" must never be read as "passed".

**Keep three things apart:** valid syntax (Python can read it) ≠ runs successfully (no error while running) ≠ meets the assignment (matches what the assignment asks for).

We chose it because it happens to use all three kinds of things (and you can also combine them):

| What the helper needs | Example | What we use |
| --- | --- | --- |
| Steps controlled by a flow | Check syntax first; report errors; only on a pass, write feedback and check its format | A **workflow** defined in code (LangGraph, Session 1) |
| Things that need judgment | How to word the feedback, whether to actually run the code and see | Leave it to the **AI** (agent, Session 1) |
| Ways of working and rules | Should the feedback be gentle or strict, what a good variable name looks like | A **handbook** written for the AI to read (Skill, Session 2). Rules that code can check (such as variable naming) can also come with a script or go into the workflow; 2-B compares them |

👉 **This notebook covers:** A helper (agent) that lets the AI decide how to review: no flow diagram, just a toolbox (run the homework, check indentation, read the assignment, plus a "homework repo" connected through an MCP socket), so you can watch it pick tools and see what can go wrong.

<details><summary>Open this when you need to look up a component name (no need to memorize it)</summary>

| Component | What it does, in plain words |
| --- | --- |
| `check_syntax` (check syntax) | Only checks whether the code's "grammar" is right, for example whether brackets are closed or a colon is missing. It **does not run the code** |
| `explain_error` (explain the error) | When the syntax is wrong, tells the student in one sentence which line is wrong |
| `give_feedback` (write feedback) | Once a model is connected in 1-A, asks the AI to write "Strengths + Improvements" feedback; the next station checks whether it matches the format |
| `check_format` (check format) | Checks the order of the two markers, that each part is non-empty, and the total character count. It does not verify that the feedback content is correct |
| Tools: `run_code`, `check_indent`, `read_spec` (from 1-B on); `check_naming` (from 2-B on) | Run the homework under teaching limits, check indentation, read the assignment requirements, check variable names |
| Handbooks: `friendly-feedback`, `strict-review`; `naming-check` (2-B), `detailed-feedback` (the 2-C security experiment) | Tell the AI whether feedback should be gentle and encouraging or strict; the naming-check handbook comes with a script; the last one hides a suspicious instruction so you can practice checking an outside handbook |

</details>

**Reminder:** run "0. Setup" first, then follow each part: setup cell → predict → run → compare.
`%%writefile` means "save this cell's content as a file"; edits only take effect after you run the cell. Re-running a setup cell may overwrite the handbooks, so save your own edits first.

In [ ]:
#@title 0. Setup (run it the first time you open this notebook, or after the runtime is reset)
#@markdown **Model source**: `colab` = the model built into Colab (no API key needed; availability and quota depend on your account); `gemini` = the instructor's Gemini API key (the backup option)
model_source = "colab"  #@param ["colab", "gemini"]
#@markdown If you pick `gemini`: we recommend adding `GOOGLE_API_KEY` under 🔑 "Secrets" in the left sidebar. You can also paste the key below (⚠️ if you paste it, do not share this notebook)
INSTRUCTOR_API_KEY = ""  #@param {type:"string"}
#@markdown Advanced (you usually don't need to change these): set a model name, or leave it empty to use the default
colab_model_name = ""  #@param {type:"string"}
gemini_model_name = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. Install packages with uv (limited to compatible version ranges; time depends on your network)
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ Installing LangGraph / LangChain with uv ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4" "mcp>=1.20,<2" "langchain-mcp-adapters>=0.3,<0.4"', shell=True, check=True)

# 2. Write the shared files this notebook uses
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_files = {
    'llm.py': r'''"""Model adapter: wraps either "Colab's built-in model" or "the instructor's Gemini API key" into one model object.

Students don't need to read this file. Every later cell only writes model.invoke(...) and model.bind_tools(...),
and it works the same no matter which provider is behind it.

- colab : google.colab.ai.generate_text (works with a free account, no API key needed)
- gemini: langchain_google_genai.ChatGoogleGenerativeAI (needs GOOGLE_API_KEY)

generate_text takes only one piece of text and returns only one piece of text. It has no native tool calling.
So ColabAIChat adds a small adapter:
1. It joins the messages into one text (system instructions, user, AI, and tool results each get a header)
2. When tools are bound, it appends the tool list and asks the model to "reply with a single line of JSON if you need a tool"
3. It parses the model's JSON into tool_calls, in exactly the same format as native tool calling
This lets you see that the model only ever "says" which tool it wants.
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

SETTINGS_FILE = Path(__file__).with_name(".llm.json")
DEFAULT_GEMINI_MODEL = "gemini-flash-latest"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: about 4 characters per token for English text.
    This is only a guess. A real tokenizer can count noticeably more or fewer."""
    return max(1, round(len(text) / 4))


# ---------- Join the messages into one text ----------

def _describe_tools(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params}): {desc}")
    return "\n".join(lines)


def _messages_to_text(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"=== System instructions ===\n{m.text}")
        elif m.type == "human":
            parts.append(f"=== User ===\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("=== AI ===\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"=== AI ===\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "tool"
            parts.append(f"=== Result of tool {name} ===\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "This time you must call at least one tool.\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"This time you must call {tool_choice}.\n"
        parts.append(
            "=== Tools you can use ===\n"
            f"{_describe_tools(tools)}\n\n"
            "=== Reply rules ===\n"
            f"{must}"
            "If you need to use a tool, reply with a single line of JSON only. Add no other text and no ``` marks:\n"
            '{"tool_calls": [{"name": "tool name", "args": {"parameter name": "parameter value"}}]}\n'
            "If you don't need a tool (for example, you already have enough information), just answer in plain text and output no JSON."
        )
    parts.append("=== AI ===")
    return "\n\n".join(parts)


def _parse_reply(text: str, tools: list[dict] | None) -> AIMessage:
    """If the model replied with a JSON note, turn it into tool_calls. Otherwise treat it as plain text."""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """Wraps google.colab.ai.generate_text as a LangChain chat model."""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _messages_to_text(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _parse_reply(text, tools)
        message.usage_metadata = {
            "input_tokens": estimate_tokens(prompt),
            "output_tokens": estimate_tokens(text),
            "total_tokens": estimate_tokens(prompt) + estimate_tokens(text),
        }
        message.response_metadata["token_count"] = "estimate (Colab's built-in model does not report usage)"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- Public: settings and getting the model ----------

def load_settings() -> dict:
    try:
        return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_settings(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """Remember which model provider was chosen (the API key is not saved)."""
    SETTINGS_FILE.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or DEFAULT_GEMINI_MODEL},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _find_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = load_settings()
    # In the cloud there is no settings file: use the environment variable WORKSHOP_PROVIDER=gemini (the key is read from GOOGLE_API_KEY)
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _find_api_key()
        if not key:
            raise RuntimeError(
                "You chose gemini but no API key was found. Add GOOGLE_API_KEY under 🔑 Secrets on the left, "
                "or paste the key from the instructor into the form in \"0. Environment setup\", then rerun that cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or DEFAULT_GEMINI_MODEL, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai only works on Google Colab. If you are not on Colab, choose gemini instead.") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""Small helpers shared by the workshop notebooks (students don't need to read this).

- prepare()                     : the first cell of every part calls it to reconnect the model and reload edited files
- sample_homework               : the sample homework in the form dropdown
- record() / print_history()    : save the results that experiment cells name, then list them by title in the last cell
- show_graph()                  : draw a LangGraph as a diagram
- save_to_drive() / download_zip()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, estimate_tokens  # noqa: F401  (so notebooks can write: from workshop import estimate_tokens)

__all__ = [
    "BASE", "prepare", "sample_homework", "load_homework", "record", "print_state", "print_order_slip", "print_history", "clear_history",
    "ModelCallCounter", "show_graph", "save_to_drive", "download_zip", "estimate_tokens",
]

BASE = Path(__file__).resolve().parent
HISTORY_FILE = BASE / "history.jsonl"

sample_homework = {
    "Missing parenthesis": "print('hi'\n",
    "Correct program": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "Empty": "",
    "Variable typo": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "Messy indentation": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "Inconsistent naming": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def load_homework(name: str) -> str:
    """When the form says "My homework", read /content/review-bot/my_homework.py. Otherwise take it from the samples."""
    if name.startswith("My homework"):
        path = BASE / "my_homework.py"
        if not path.exists():
            raise FileNotFoundError('There is no "My homework" yet. Run the cell above that has %%writefile ... my_homework.py first.')
        return path.read_text(encoding="utf-8")
    return sample_homework[name]


def prepare():
    """The start of every part: put review-bot on the import path, reload edited files, and return the model."""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _make_mcp_stdio_work_in_notebook()
    return get_model()


def _make_mcp_stdio_work_in_notebook() -> None:
    """mcp's stdio_client connects the MCP server's error output to sys.stderr.
    On Colab (Linux), sys.stderr has a real file descriptor, so this works. Jupyter on Windows has none, so we redirect to a log file."""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State history ----------

def _serializable(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serializable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def record(title: str, content) -> None:
    """Save this result (usually a State) into history.jsonl."""
    row = {"time": _dt.datetime.now().strftime("%H:%M:%S"), "title": title, "content": _serializable(content)}
    with HISTORY_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _short(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"... ({len(text) - n} more characters)"


def print_state(state: dict, indent: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{indent}{k}: ({len(v)} messages)")
            for m in v:
                m = _serializable(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{indent}  [{m['type']}] calls {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{indent}  [{m.get('type')}] {_short(m.get('content', ''), 200)}")
        else:
            print(f"{indent}{k}: {_short(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def print_order_slip(step_no: int, state: dict, prev_state: dict | None = None, skip_keys: tuple = ("code",)) -> None:
    """stream_mode="values" gives the whole order slip at every step. This also marks which fields this step newly wrote."""
    if not prev_state:
        note = "start: blank order slip"
    else:
        changed_keys = [k for k in state if state.get(k) != prev_state.get(k)]
        note = "newly written: " + (", ".join(changed_keys) if changed_keys else "(nothing changed)")
    print(f"Step {step_no} ({note})")
    print("  ", {k: (_short(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in skip_keys})


def print_history() -> None:
    if not HISTORY_FILE.exists():
        print("There are no records yet. Run a few parts above first, then come back to this cell.")
        return
    rows = [json.loads(line) for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 {len(rows)} records (saved in {HISTORY_FILE})\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['time']}] {row['title']} =====")
        content = row["content"]
        if isinstance(content, dict):
            print_state(content)
        elif isinstance(content, list):
            for j, item in enumerate(content, 1):
                print(f"  --- step {j} ---")
                if isinstance(item, dict):
                    print_state(item, "    ")
                else:
                    print(f"    {_short(item)}")
        else:
            print(f"  {_short(content, 1000)}")
        print()


def clear_history() -> None:
    HISTORY_FILE.unlink(missing_ok=True)


# ---------- Count how many times the model was called ----------

class ModelCallCounter(BaseCallbackHandler):
    """Put it in config={"callbacks": [counter]}. It adds 1 every time the model is called."""

    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.count += 1


# ---------- Draw the graph ----------

def show_graph(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("(The drawing service is unreachable for now, so here is a text version. You can paste it into https://mermaid.live to see the diagram.)\n")
        print(g.draw_mermaid())


# ---------- Saving ----------

def save_to_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, destination, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ Saved to Google Drive: {destination}")
    print('   (Save the notebook itself with the menu above: "File → Save a copy in Drive")')


def download_zip() -> None:
    from google.colab import files

    name = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(name), "zip", BASE.parent, BASE.name)
    print(f"✅ Packed: {zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""The Homework Helper's toolbox: run_code, check_indent, read_spec, check_naming."""
''',
    'tools/run_code.py': r'''"""run_code: runs a student's homework under teaching restrictions.

A student's homework could contain anything (import os to delete files, an infinite loop), so this tool has two execution limits plus one readability feature:
1. A check before running: only a few standard modules may be imported (math, random, re, ...). os / sys / subprocess and the like are always blocked.
   eval, exec, open, input and attributes that start with an underscore (often used to get around restrictions) are not allowed either
2. The code runs in a separate child process, and it is stopped by force when the time runs out (so an infinite loop can be stopped too)
3. It returns only one line of result (which line, what error), which is easier for people and models to read. This is not a security limit

⚠️ This is a teaching fence. For a real product, use a real sandbox (a container, RestrictedPython, e2b, Modal).
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

BLOCKED_MODULES = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
ALLOWED_MODULES = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
BLOCKED_FUNCTIONS = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
BLOCKED_ATTRIBUTES = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
TIME_LIMIT = 3
MAX_OUTPUT_CHARS = 800

# The code in the child process that really runs the homework: it uses a "shrunken builtins", and import lets only the allowlist through
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("This tool does not allow import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "homework.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "homework.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _precheck(code: str) -> str | None:
    """A static check before running. If there is a problem, return a one-line explanation. Otherwise return None."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ Line {e.lineno} SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in BLOCKED_MODULES or root not in ALLOWED_MODULES:
                    return f"⛔ Blocked: line {node.lineno} import {name} (this tool does not allow {root})"
        elif isinstance(node, ast.Name) and node.id in BLOCKED_FUNCTIONS:
            return f"⛔ Blocked: line {node.lineno} uses {node.id} (this tool does not allow it)"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in BLOCKED_ATTRIBUTES or node.attr in BLOCKED_MODULES
        ):
            return f"⛔ Blocked: line {node.lineno} accesses .{node.attr} (this tool does not allow it)"
    return None


def run_homework(code: str, time_limit_seconds: float = TIME_LIMIT) -> str:
    """The function that does the real work: check → run in a child process → return one line of result."""
    if not code.strip():
        return "❌ The homework is empty, there is nothing to run"
    blocked = _precheck(code)
    if blocked:
        return blocked
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(ALLOWED_MODULES)), json.dumps(sorted(BLOCKED_FUNCTIONS))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=time_limit_seconds, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ Ran longer than {time_limit_seconds:g} seconds and was stopped by force (there may be an infinite loop)"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ The run failed (the program did not end normally; it may have used too much memory)"
    stdout = result.get("stdout", "")
    if len(stdout) > MAX_OUTPUT_CHARS:
        stdout = stdout[:MAX_OUTPUT_CHARS] + "\n... (output too long, the rest is omitted)"
    if result["ok"]:
        return f"✅ Ran successfully, output:\n{stdout}" if stdout else "✅ Ran successfully (nothing was printed)"
    where = f"line {result['line']} " if result.get("line") else ""
    output = f"\n(printed before the error: {stdout.strip()})" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{output}"


@tool
def run_code(code: str) -> str:
    """Run a piece of Python homework under teaching restrictions and return the output or the error message.

    When to use it: when you want to know what the homework actually prints when it runs, or whether it errors.
    Parameter code: the complete Python code (a string).
    Returns: one line of result, for example "✅ Ran successfully, output: 5" or "❌ line 3 NameError: ...".
    Limits: it can only import a few standard modules such as math, random and re; it cannot use input() or open(); it is stopped after 3 seconds.
    """
    return run_homework(code)
''',
    'tools/check_indent.py': r'''"""check_indent: checks indentation and line length.

It deliberately checks ONLY indentation and line length, and never touches variable naming. The naming check is left to Session 2's 2-B,
where it is used to compare "written in the handbook text" with "written as a script".
"""

from langchain.tools import tool

MAX_LINE_LENGTH = 79


def check_indentation(code: str, limit: int = MAX_LINE_LENGTH) -> str:
    problems = []
    for i, line in enumerate(code.splitlines(), start=1):
        if not line.strip():
            continue
        leading_spaces = line[: len(line) - len(line.lstrip())]
        if "\t" in leading_spaces:
            problems.append(f"Line {i}: indented with a Tab; use 4 spaces instead")
        elif len(leading_spaces) % 4 != 0:
            problems.append(f"Line {i}: indented {len(leading_spaces)} spaces; use a multiple of 4")
        if len(line) > limit:
            problems.append(f"Line {i}: {len(line)} characters, over the limit of {limit}")
    return "\n".join(problems) if problems else "OK: indentation and line length look fine"


@tool
def check_indent(code: str) -> str:
    """Check the indentation of a Python homework (is it a multiple of 4, are Tabs mixed in) and the length of each line (limit 79 characters).

    When to use it: when you want to confirm the homework's layout is tidy. It does not check variable naming and does not run the code.
    Parameter code: the complete Python code (a string).
    Returns: one line per problem, for example "Line 2: indented 2 spaces; use a multiple of 4". If everything is fine it returns "OK: ...".
    """
    return check_indentation(code)
''',
    'tools/read_spec.py': r'''"""read_spec: reads the assignment requirements for this question.

The requirements are stored in review-bot/spec.md. When the notebook form changes the assignment, this file is rewritten.
"""

from pathlib import Path

from langchain.tools import tool

SPEC_FILE = Path(__file__).resolve().parent.parent / "spec.md"


def read_spec_text() -> str:
    if not SPEC_FILE.exists():
        return "(No assignment requirements were provided for this question)"
    return SPEC_FILE.read_text(encoding="utf-8").strip() or "(No assignment requirements were provided for this question)"


@tool
def read_spec() -> str:
    """Read the assignment requirements for this question (what the assignment says and what to hand in).

    When to use it: before judging whether the homework meets the assignment, for example whether the function name or the output format is right.
    It takes no parameters.
    Returns: the full text of the assignment requirements.
    """
    return read_spec_text()
''',
    'agent.py': r'''"""1-B: the Homework Helper as an agent. The route is decided by the AI.

Agent = model + tools + loop.
- make_agent()          : the standard LangChain 1.0 way, create_agent, which builds an agent in one line
- make_expanded_agent() : the same agent assembled by hand with LangGraph, so you can see the loop underneath
- pretty_trace()        : prints every step so you can see which tool it picked
"""

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from tools.check_indent import check_indent
from tools.read_spec import read_spec
from tools.run_code import run_code

base_system_prompt = (
    "You are a Python homework helper. The students are beginners who just learned Python, so give feedback in English with a gentle tone. "
    "You have a few tools. You decide whether to use one, which one, and how many times. "
    'Write the feedback in two paragraphs: the first starts with "Strengths:" and the second starts with "Improvements:".'
)
helper_tools = [run_code, check_indent, read_spec]


def make_agent(model, system_prompt: str = base_system_prompt, tools=None):
    return create_agent(model, tools=tools or helper_tools, system_prompt=system_prompt)


def make_expanded_agent(model, system_prompt: str = base_system_prompt, tools=None):
    """The core loop for teaching: it expands how the model and the tools interact. It is not the full implementation of create_agent."""
    tools = tools or helper_tools
    model_with_tools = model.bind_tools(tools)

    def ask_model(state: MessagesState) -> dict:
        messages = [SystemMessage(system_prompt)] + state["messages"]
        return {"messages": [model_with_tools.invoke(messages)]}

    builder = StateGraph(MessagesState)
    builder.add_node("ask_model", ask_model)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "ask_model")
    # tools_condition: if the last message has tool_calls go to tools, otherwise go to END
    builder.add_conditional_edges("ask_model", tools_condition)
    builder.add_edge("tools", "ask_model")  # after tools run, loop back to ask_model
    return builder.compile()


def homework_message(homework: str) -> str:
    return f"Please look at this homework:\n```python\n{homework}\n```"


def pretty_trace(graph, user_message: str, config: dict | None = None) -> list[dict]:
    events = []
    print(f"USER: {user_message}\n")
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    stream = graph.stream(inputs, config=config or {}, stream_mode="updates")
    for step, event in enumerate(stream, start=1):
        events.append(event)
        for node_name, node_state in event.items():
            print(f"--- Step {step} | node: {node_name} ---")
            for msg in (node_state or {}).get("messages", []):
                role = type(msg).__name__
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for call in tool_calls:
                        print(f"  The model decided to call -> {call['name']}({call['args']})")
                else:
                    text = msg.text if hasattr(msg, "text") else str(msg.content)
                    if len(text) > 400:
                        text = text[:400] + "..."
                    print(f"  [{role}] {text}")
            print()
    return events


def tool_sequence(messages_or_events) -> list[str]:
    """Pull the order of tool calls out of messages (or out of the events from pretty_trace)."""
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    return [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]


def final_reply(messages_or_events) -> str:
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    for m in reversed(messages):
        if m.type == "ai" and not getattr(m, "tool_calls", None):
            return m.text
    return ""
''',
    'spec.md': r'''Write a function add(a, b) that returns the sum of two numbers, and use print(add(2, 3)) to print 5.
''',
}
for rel, content in _files.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# Make Python look again for the files we just wrote (if another cell ran first, Python remembers "this folder does not exist")
import importlib
importlib.invalidate_caches()

# 3. Remember the model source (the API key is not saved) and test the connection
import llm
llm.save_settings(model_source, colab_model_name.strip(), gemini_model_name.strip())
if INSTRUCTOR_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = INSTRUCTOR_API_KEY.strip()
from workshop import *
model = prepare()

reply = model.invoke("Say hello in one sentence to a classmate who just learned Python").text
print(f"✅ Setup complete. Model source: {model_source}")
print(f"🤖 The model says: {reply.strip()}")

### ✏️ (Optional) Bring in your own homework

The take-home assignment asks you to run **a piece of homework you wrote yourself**. Replace the code in the cell below with your homework and run it.
After that, choose **My homework** in the form's "homework" field and the helper will use it. You can skip this cell in class.
`run_code` does not allow `input()` or `open()`, and only a few standard modules can be imported. If your homework uses `input()`, change it to a fixed value first (for example `n = 5`).

In [ ]:
%%writefile /content/review-bot/my_homework.py
# Replace this with a piece of Python homework you wrote yourself, then remember to run this cell
def average(scores):
    total = 0
    for s in scores:
        total = total + s
    return total / len(scores)

print(average([90, 85, 77]))


---
## Part 1: The toolbox, and run_code's guardrails

> 📍 **Matching slides**: slide 31 "A different approach: no flow diagram, just a toolbox", slide 32 "Try the tools first, then hand them to the agent"

First, prepare the agent's toolbox.

The next three cells write out three tools (no changes needed, just run them). Each tool is an ordinary Python function
with one `@tool` line on top, and once you add it to the agent's tool list, the model can ask to use it.

`run_code` is a tool that **really runs code**, and anything can be in a student's homework, so first look at two execution limits and one design that makes things easier to read:
1. Before running, it checks the allowed syntax and modules, and blocks the dangerous operations tested in this session
2. A time limit (timeout): it is force-stopped after 3 seconds, so even an infinite loop stops
3. It shrinks an error message to a single line so people and the model can read it easily; this is not a safety limit

### 🤔 Prediction questions
Below, four homeworks are passed to `run_code`: a normal one, one with a mistyped variable, one that sneaks in `import os`, and one with an infinite loop. What result will each get?

In [ ]:
%%writefile /content/review-bot/tools/run_code.py
"""run_code: runs a student's homework under teaching restrictions.

A student's homework could contain anything (import os to delete files, an infinite loop), so this tool has two execution limits plus one readability feature:
1. A check before running: only a few standard modules may be imported (math, random, re, ...). os / sys / subprocess and the like are always blocked.
   eval, exec, open, input and attributes that start with an underscore (often used to get around restrictions) are not allowed either
2. The code runs in a separate child process, and it is stopped by force when the time runs out (so an infinite loop can be stopped too)
3. It returns only one line of result (which line, what error), which is easier for people and models to read. This is not a security limit

⚠️ This is a teaching fence. For a real product, use a real sandbox (a container, RestrictedPython, e2b, Modal).
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

BLOCKED_MODULES = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
ALLOWED_MODULES = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
BLOCKED_FUNCTIONS = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
BLOCKED_ATTRIBUTES = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
TIME_LIMIT = 3
MAX_OUTPUT_CHARS = 800

# The code in the child process that really runs the homework: it uses a "shrunken builtins", and import lets only the allowlist through
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("This tool does not allow import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "homework.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "homework.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _precheck(code: str) -> str | None:
    """A static check before running. If there is a problem, return a one-line explanation. Otherwise return None."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ Line {e.lineno} SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in BLOCKED_MODULES or root not in ALLOWED_MODULES:
                    return f"⛔ Blocked: line {node.lineno} import {name} (this tool does not allow {root})"
        elif isinstance(node, ast.Name) and node.id in BLOCKED_FUNCTIONS:
            return f"⛔ Blocked: line {node.lineno} uses {node.id} (this tool does not allow it)"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in BLOCKED_ATTRIBUTES or node.attr in BLOCKED_MODULES
        ):
            return f"⛔ Blocked: line {node.lineno} accesses .{node.attr} (this tool does not allow it)"
    return None


def run_homework(code: str, time_limit_seconds: float = TIME_LIMIT) -> str:
    """The function that does the real work: check → run in a child process → return one line of result."""
    if not code.strip():
        return "❌ The homework is empty, there is nothing to run"
    blocked = _precheck(code)
    if blocked:
        return blocked
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(ALLOWED_MODULES)), json.dumps(sorted(BLOCKED_FUNCTIONS))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=time_limit_seconds, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ Ran longer than {time_limit_seconds:g} seconds and was stopped by force (there may be an infinite loop)"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ The run failed (the program did not end normally; it may have used too much memory)"
    stdout = result.get("stdout", "")
    if len(stdout) > MAX_OUTPUT_CHARS:
        stdout = stdout[:MAX_OUTPUT_CHARS] + "\n... (output too long, the rest is omitted)"
    if result["ok"]:
        return f"✅ Ran successfully, output:\n{stdout}" if stdout else "✅ Ran successfully (nothing was printed)"
    where = f"line {result['line']} " if result.get("line") else ""
    output = f"\n(printed before the error: {stdout.strip()})" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{output}"


@tool
def run_code(code: str) -> str:
    """Run a piece of Python homework under teaching restrictions and return the output or the error message.

    When to use it: when you want to know what the homework actually prints when it runs, or whether it errors.
    Parameter code: the complete Python code (a string).
    Returns: one line of result, for example "✅ Ran successfully, output: 5" or "❌ line 3 NameError: ...".
    Limits: it can only import a few standard modules such as math, random and re; it cannot use input() or open(); it is stopped after 3 seconds.
    """
    return run_homework(code)


In [ ]:
%%writefile /content/review-bot/tools/check_indent.py
"""check_indent: checks indentation and line length.

It deliberately checks ONLY indentation and line length, and never touches variable naming. The naming check is left to Session 2's 2-B,
where it is used to compare "written in the handbook text" with "written as a script".
"""

from langchain.tools import tool

MAX_LINE_LENGTH = 79


def check_indentation(code: str, limit: int = MAX_LINE_LENGTH) -> str:
    problems = []
    for i, line in enumerate(code.splitlines(), start=1):
        if not line.strip():
            continue
        leading_spaces = line[: len(line) - len(line.lstrip())]
        if "\t" in leading_spaces:
            problems.append(f"Line {i}: indented with a Tab; use 4 spaces instead")
        elif len(leading_spaces) % 4 != 0:
            problems.append(f"Line {i}: indented {len(leading_spaces)} spaces; use a multiple of 4")
        if len(line) > limit:
            problems.append(f"Line {i}: {len(line)} characters, over the limit of {limit}")
    return "\n".join(problems) if problems else "OK: indentation and line length look fine"


@tool
def check_indent(code: str) -> str:
    """Check the indentation of a Python homework (is it a multiple of 4, are Tabs mixed in) and the length of each line (limit 79 characters).

    When to use it: when you want to confirm the homework's layout is tidy. It does not check variable naming and does not run the code.
    Parameter code: the complete Python code (a string).
    Returns: one line per problem, for example "Line 2: indented 2 spaces; use a multiple of 4". If everything is fine it returns "OK: ...".
    """
    return check_indentation(code)


In [ ]:
%%writefile /content/review-bot/tools/read_spec.py
"""read_spec: reads the assignment requirements for this question.

The requirements are stored in review-bot/spec.md. When the notebook form changes the assignment, this file is rewritten.
"""

from pathlib import Path

from langchain.tools import tool

SPEC_FILE = Path(__file__).resolve().parent.parent / "spec.md"


def read_spec_text() -> str:
    if not SPEC_FILE.exists():
        return "(No assignment requirements were provided for this question)"
    return SPEC_FILE.read_text(encoding="utf-8").strip() or "(No assignment requirements were provided for this question)"


@tool
def read_spec() -> str:
    """Read the assignment requirements for this question (what the assignment says and what to hand in).

    When to use it: before judging whether the homework meets the assignment, for example whether the function name or the output format is right.
    It takes no parameters.
    Returns: the full text of the assignment requirements.
    """
    return read_spec_text()


In [ ]:
#@title ▶ Run: try run_code's limits and error messages
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from tools.run_code import run_code

four_homeworks = {
    "Normal homework": "print(sum([1, 2, 3]))",
    "Mistyped variable": "total = 1 + 2\nprint(totla)",
    "Sneaky import os": "import os\nos.remove('important_file.txt')",
    "Infinite loop": "while True:\n    pass",
}
results = {}
for name, homework in four_homeworks.items():
    results[name] = run_code.invoke({"code": homework})
    print(f"[{name}]\n{homework}\n-> {results[name]}\n")
record("1-B Part 1: run_code guardrails", results)

### 👀 What did you see?

- The normal homework prints `6`; the mistyped variable returns just one line like "Line 2 NameError...", not a long traceback.
- `import os` is blocked **before** running, and not a single file was touched; the infinite loop is force-stopped after 3 seconds.
- Running outside code needs isolation and permission limits: the model can be influenced by its input and may ask `run_code` to do something inappropriate (Session 2's 2-C comes back to this).
  The guardrails here are a simplified teaching version and not a complete sandbox. A real product needs a real sandbox (containers, e2b, Modal, and so on).

---
## Part 2: What tool information does the model get?

> 📍 **Matching slides**: slide 33 "What a tool interface tells the model"

The last part wrote three tools. A tool interface doesn't automatically send the function's implementation; so how does the model know what each tool can do?

`@tool` wraps the function name, the parameters, and the explanation in triple quotes at the top of the function (called the docstring) into a **description** and hands it to the model.
The model consults that description, the task and the conversation to decide whether to ask for a tool.

### 🤔 Prediction questions
Can the model see the code inside `run_code` that forbids `import os`? How does it know `read_spec` is for reading the assignment?

In [ ]:
#@title ▶ Run: print the descriptions of the three tools
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from tools.check_indent import check_indent
from tools.read_spec import read_spec
from tools.run_code import run_code

for t in [run_code, check_indent, read_spec]:
    print("name:", t.name)
    print("description:", t.description)
    print("args:", t.args)
    print("-" * 60)

### 👀 What did you see?

- The tool interface here provides `name`, `description` (the docstring) and `args`; it does not automatically hand the function's implementation to the model.
- A good docstring says four things: **what it does, when to call it, what the parameters mean, and what it returns**.
  If it only says "run code" and not when to use it, the model may skip it or misuse it.
- The model cannot see the code that blocks `import os`, but it can see the line in the docstring saying "Limits: only a few standard modules such as math, random and re can be imported...".
- Skills in Session 2 also have a name and a description, which help the model decide whether to read the full guidance.

---
## Part 3: Build an agent in one line, and see what it picks at each step

> 📍 **Matching slides**: slide 34 "An agent in one line: create_agent", slide 35 "Reading an agent's execution trace"

The first two parts prepared the toolbox and its descriptions. This part puts the model and the toolbox together to make an agent, and shows what it picks at each step while reviewing a homework.

The next cell writes out `agent.py`: its `make_agent` is just the one line `create_agent(model, tools=..., system_prompt=...)`,
and this session uses that interface:

- `tools`: the toolbox (three local tools)
- `system_prompt`: the instruction placed at the very start of the conversation and carried along every time, for example "You are a Python homework helper, and your tone should be gentle"
- `pretty_trace`: prints out each step of the agent so you can see the tool requests the model proposes and the tool results; it is not the model's internal thinking

### 🤔 Prediction questions (write on your worksheet)
Pick the homework "Variable typo" (`return summ`, but the variable is called `total`), and the assignment is "write an add(a, b)".
1. Will it use a tool? If so, which one **first**?
2. How many times in total? Will it use all three? (It may also use none and just reply.)

In [ ]:
%%writefile /content/review-bot/agent.py
"""1-B: the Homework Helper as an agent. The route is decided by the AI.

Agent = model + tools + loop.
- make_agent()          : the standard LangChain 1.0 way, create_agent, which builds an agent in one line
- make_expanded_agent() : the same agent assembled by hand with LangGraph, so you can see the loop underneath
- pretty_trace()        : prints every step so you can see which tool it picked
"""

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from tools.check_indent import check_indent
from tools.read_spec import read_spec
from tools.run_code import run_code

base_system_prompt = (
    "You are a Python homework helper. The students are beginners who just learned Python, so give feedback in English with a gentle tone. "
    "You have a few tools. You decide whether to use one, which one, and how many times. "
    'Write the feedback in two paragraphs: the first starts with "Strengths:" and the second starts with "Improvements:".'
)
helper_tools = [run_code, check_indent, read_spec]


def make_agent(model, system_prompt: str = base_system_prompt, tools=None):
    return create_agent(model, tools=tools or helper_tools, system_prompt=system_prompt)


def make_expanded_agent(model, system_prompt: str = base_system_prompt, tools=None):
    """The core loop for teaching: it expands how the model and the tools interact. It is not the full implementation of create_agent."""
    tools = tools or helper_tools
    model_with_tools = model.bind_tools(tools)

    def ask_model(state: MessagesState) -> dict:
        messages = [SystemMessage(system_prompt)] + state["messages"]
        return {"messages": [model_with_tools.invoke(messages)]}

    builder = StateGraph(MessagesState)
    builder.add_node("ask_model", ask_model)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "ask_model")
    # tools_condition: if the last message has tool_calls go to tools, otherwise go to END
    builder.add_conditional_edges("ask_model", tools_condition)
    builder.add_edge("tools", "ask_model")  # after tools run, loop back to ask_model
    return builder.compile()


def homework_message(homework: str) -> str:
    return f"Please look at this homework:\n```python\n{homework}\n```"


def pretty_trace(graph, user_message: str, config: dict | None = None) -> list[dict]:
    events = []
    print(f"USER: {user_message}\n")
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    stream = graph.stream(inputs, config=config or {}, stream_mode="updates")
    for step, event in enumerate(stream, start=1):
        events.append(event)
        for node_name, node_state in event.items():
            print(f"--- Step {step} | node: {node_name} ---")
            for msg in (node_state or {}).get("messages", []):
                role = type(msg).__name__
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for call in tool_calls:
                        print(f"  The model decided to call -> {call['name']}({call['args']})")
                else:
                    text = msg.text if hasattr(msg, "text") else str(msg.content)
                    if len(text) > 400:
                        text = text[:400] + "..."
                    print(f"  [{role}] {text}")
            print()
    return events


def tool_sequence(messages_or_events) -> list[str]:
    """Pull the order of tool calls out of messages (or out of the events from pretty_trace)."""
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    return [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]


def final_reply(messages_or_events) -> str:
    messages = []
    for item in messages_or_events:
        if isinstance(item, dict):
            for node_state in item.values():
                messages += (node_state or {}).get("messages", [])
        else:
            messages.append(item)
    for m in reversed(messages):
        if m.type == "ai" and not getattr(m, "tool_calls", None):
            return m.text
    return ""


In [ ]:
#@title ▶ Run: the agent made by create_agent, and each of its steps
homework = "Variable typo"  #@param ["Variable typo", "Correct program", "Messy indentation", "Missing parenthesis", "My homework"]
spec = "Write a function add(a, b) that returns the sum of two numbers, and use print(add(2, 3)) to print 5."  #@param {type:"string"}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import agent

(BASE / "spec.md").write_text(spec, encoding="utf-8")   # read_spec reads this file
helper = agent.make_agent(model)
events = agent.pretty_trace(helper, agent.homework_message(load_homework(homework)))
print("Order in which tools were used this time:", " -> ".join(agent.tool_sequence(events)) or "(no tools used)")
record(f"1-B Part 3: create_agent ({homework})", events)

### 👀 What did you see?

- Each "node: model" is one model response; "node: tools" is your program running the tool it picked. One back-and-forth is one turn of the loop from 1-0.
- The `AIMessage` at the last step did not call a tool again, so the loop ended.
- If there is only one "node: model" and the output ends with "(no tools used)", the model replied without requesting any tool, which is also a valid observation. In that case, any error mentioned in the feedback was the model's guess from reading the code, not something `run_code` actually ran.
- Compare with your prediction: which did it use first? Was there a tool it never used at all? The next part will show that this order can differ every time.
- `invoke` gives you only the end result; `stream` (which `pretty_trace` uses) lays out the process.

---
## Part 4: Same homework three times, will the order be the same?

> 📍 **Matching slides**: slide 36 "Same homework, three runs: same order?"

The last part watched the agent's route once. If you run the same homework three more times, will the route be the same?

This is the biggest difference between an agent and a workflow: **the route isn't one you drew, it's one it decides in the moment**.
This part also lets you change the **system prompt** (the instruction placed at the very start every time) by adding a sentence, and watch how the new instruction affects the order of tools.

### 🤔 Prediction questions (two rounds)
1. First leave "system prompt extra" blank and run. Will the order of tool use be the same across the three runs?
2. Then change it to **Always read the assignment requirements first** and run. Did the order change this time? Are all three the same now?

In [ ]:
#@title ▶ Run: run the same homework three times and compare the order of tool use
homework = "Variable typo"  #@param ["Variable typo", "Correct program", "Messy indentation", "Missing parenthesis", "My homework"]
system_prompt_extra = ""  #@param ["", "Always read the assignment requirements first", "Always run the homework and look at the result first", "Do not use any tools"] {allow-input: true}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import agent

system_prompt = agent.base_system_prompt + system_prompt_extra
print("system prompt:", system_prompt, "\n")
helper = agent.make_agent(model, system_prompt=system_prompt)

three_runs = []
for i in range(3):
    result = helper.invoke({"messages": [{"role": "user", "content": agent.homework_message(load_homework(homework))}]})
    sequence = agent.tool_sequence(result["messages"])
    three_runs.append(sequence)
    print(f"========== Run {i + 1} ==========")
    print("Order of tool use:", " -> ".join(sequence) or "(no tools used, went straight to feedback)")
    print(result["messages"][-1].text[:300])
    print()

print("Same order in all three runs?", "Yes" if all(x == three_runs[0] for x in three_runs) else "No, they differ")
record(f"1-B Part 4: three runs (extra sentence: {system_prompt_extra or 'none'})", {"tool order in the three runs": three_runs})

### 👀 What did you see?

- The three runs may be the same or different. First record the actual tool order and any checks that were skipped. The same result now does not mean it will be the same later; if a necessary step is skipped, that is still a problem for the system to handle.
- After adding "Always read the assignment requirements first", see whether the first step becomes `read_spec`. **Do not treat an instruction as a guarantee of execution**:
  a prompt can influence the choice, but a check that must run should be arranged by the flow, and its result should be verified.
- So the next thing to work out is: which steps does the flow control, and which are left to AI judgment (Stop 4 on the slides).

---
## Part 5: It can go in circles: recursion limit

> 📍 **Matching slides**: slide 37 "If a tool keeps failing, will the agent stop?", slide 38 "Set a limit, then work out why it stopped"

The last part showed that the agent's route is decided by the model and may change. The worst case is that it keeps going in circles and can't stop by itself.

The cell below gives the agent a tool `always_fail` that **deliberately always fails**, then tells it "don't stop until it succeeds".
In the code we set `recursion_limit=25` to limit the number of graph steps. This is not "call tools at most 25 times"; model nodes also use up steps. The model may also stop by itself first.

### 🤔 Prediction questions
Will the model stop by itself, or keep requesting the tool until the limit is reached? Will the number of graph steps, the number of tool requests and the number of tool results be the same?

In [ ]:
#@title ▶ Run: a tool that always fails; watch for stopping by itself or hitting the recursion limit
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.errors import GraphRecursionError

@tool
def always_fail() -> str:
    """Do something very important. If it fails, try again until it succeeds."""
    return "Execution error: RuntimeError: still not working, please try again"

looping_agent = create_agent(
    model, tools=[always_fail],
    system_prompt="Your task is to call always_fail until it succeeds. No matter how many times it fails, try again. Do not give up, and do not answer in text.",
)
tool_call_requests, tool_result_count = 0, 0
try:
    for step, event in enumerate(looping_agent.stream({"messages": [{"role": "user", "content": "Let's begin."}]},
                                               config={"recursion_limit": 25}, stream_mode="updates"), start=1):
        for node, update in event.items():
            for msg in (update or {}).get("messages", []):
                if getattr(msg, "tool_calls", None):
                    for call in msg.tool_calls:
                        if call["name"] == "always_fail":
                            tool_call_requests += 1
                        print(f"--- Step {step} | node: {node} ---  model requests -> {call['name']}({call['args']})")
                elif msg.type == "tool":
                    if getattr(msg, "name", None) == "always_fail":
                        tool_result_count += 1
                    print(f"--- Step {step} | node: {node} ---  [ToolMessage] {msg.text}")
                else:
                    print(f"--- Step {step} | node: {node} ---  [AIMessage] {msg.text[:80]}")
    print("\n(This time the model stopped by itself without reaching the limit; the tool still has not succeeded. Record this result.)")
except GraphRecursionError as e:
    print(f"\n🛑 GraphRecursionError: {str(e).splitlines()[0]}")
print(f"always_fail: the model made {tool_call_requests} tool request(s) and received {tool_result_count} tool result(s)")
record("1-B Part 5: recursion limit", {"tool_call_requests": tool_call_requests, "tool_result_count": tool_result_count})

### 👀 What did you see?

- If the model keeps calling `always_fail`, every round adds to the model-call cost; it may also stop by itself.
- If it keeps running until the step limit, you get a `GraphRecursionError`. Stopping is not success.
- Read the **number of tool requests** and the **number of tool results** separately; both count only `always_fail`. After the model proposes its last request, the step limit may be hit before a tool result comes back, so you can't treat the request count as the number of times the tool actually ran.
- The "at most 5 rounds" in 1-0, the "Maximum generations" in 1-A, and the `recursion_limit` here are all stop limits set in code; the three count different units.
- Next, see the MCP demo, then go to Stop 4 on the slides to discuss which steps the flow controls and which are left to AI judgment.

---
## Part 6 (instructor demo): Where tools come from: your own functions vs. MCP sockets

> 📍 **Matching slides**: slide 39 "Where tools come from: your own functions vs. MCP sockets"

So far we built the agent from local tools and watched its run and its stopping conditions. Now we widen where tools can come from. **The tool description doesn't have to be written by you**: tools someone else built can also be plugged in through a socket.

- **Local function**: you write it yourself and it lives in the same program (our three tools).
- **MCP**: the tools live in **another program or service** (such as GitHub) and are connected through a shared "socket" standard.
  ("Socket" refers to the shared MCP standard; the program that provides the tools is called an MCP server, like the "homework repo" in this part.)
  **MCP is not another kind of agent; it is a way to plug in "tools".**
- The cost: the more tools you give the model, the longer the tool descriptions usually get. Connecting to MCP does not mean you have to hand every tool to the model.
  (The desk = the context window, the maximum number of tokens the model can handle at once; the more descriptions there are, the less room is left for the homework and the feedback.)

This part connects to an MCP server that actually runs. The next cell writes out `deploy/homework_mcp.py`: a "homework repo" (the homework and the email in it are all fake data)
with three tools: `list_homework` (who turned in homework), `get_homework` (fetch one student's homework), and `send_email` (sends mail, but fake: it only writes to `outbox.txt`).
Open it and you will see that **it's just a few ordinary Python functions with `@mcp.tool()` on top**, running in its own program.

Then the `langchain-mcp-adapters` package plugs in the socket, we first filter out two read-only tools, and hand them to the agent together with the three local tools.
(`create_agent` is the function that "puts a model and a toolbox together into an agent", which the local agent above already used.
The `await` in the code means "wait for this to finish before moving on": MCP tools talk to another program through the socket, so they need `await`, which a Colab cell can use directly.)

### 🤔 Prediction questions
1. How does an agent use a tool that comes in through the socket, compared with your own `@tool`?
2. If you ask the agent to "look at the homework Chloe turned in and run it to see if there are errors", which tool does it call first? Will it use both an MCP tool and the local `run_code`?

In [ ]:
%%writefile /content/review-bot/deploy/homework_mcp.py
"""Homework repo MCP server: the tools live in "another program" and plug in through the MCP socket.

It's just an ordinary Python file; the functions simply have @mcp.tool() on top.
In the real world this role is played by GitHub's MCP server (fetching homework) and a mailbox MCP server (sending email);
here we use fake data instead, and "sending email" only writes to outbox.txt.

Two ways to connect:
    python homework_mcp.py              # stdio: a local socket; the agent side starts it
    python homework_mcp.py --http 8765  # Streamable HTTP: it becomes a URL, like a remote MCP
"""

import argparse
import datetime as _dt
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

OUTBOX = Path(__file__).resolve().parent.parent / "outbox.txt"

# A fake homework repo: students, emails, submitted homework
SUBMISSIONS = {
    "Alex": ("alex@example.com", "print('hi'\n"),
    "Ben": ("ben@example.com", "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n"),
    "Chloe": ("chloe@example.com", "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n"),
    "Jay": ("jay@example.com", "def CalcAverage(scoreList):\n    Total = 0\n    for s in scoreList:\n        Total += s\n"
                               "    return Total / len(scoreList)\n\nmyScores = [90, 85, 77]\nprint(CalcAverage(myScores))\n"),
    "Fiona": ("fiona@example.com", "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n"
                                   "  return total / len(scores)\n\nprint(average([90, 85, 77]))\n"),
}

mcp = FastMCP("homework_repo", log_level="WARNING")


@mcp.tool()
def list_homework() -> str:
    """List the students who have submitted homework (name and email) as a JSON string. Call this first when you need to review homework."""
    return json.dumps([{"student": name, "email": email} for name, (email, _) in SUBMISSIONS.items()], ensure_ascii=False)


@mcp.tool()
def get_homework(student: str) -> str:
    """Get the homework code one student submitted. Parameter student: the student's name (from list_homework)."""
    if student not in SUBMISSIONS:
        return f'No homework found for "{student}"'
    return SUBMISSIONS[student][1]


@mcp.tool()
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a student (irreversible: once sent, it can't be taken back). Parameters: recipient email, subject, body."""
    with OUTBOX.open("a", encoding="utf-8") as f:
        f.write(f"[{_dt.datetime.now():%H:%M:%S}] To {to} | {subject}\n{body}\n\n")
    return f"(simulated) Sent to {to}"


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--http", type=int, help="Serve over Streamable HTTP on this port (omit it to use stdio)")
    args = parser.parse_args()
    if args.http:
        mcp.settings.host = "127.0.0.1"
        mcp.settings.port = args.http
        mcp.run(transport="streamable-http")   # URL: http://127.0.0.1:<port>/mcp
    else:
        mcp.run()


In [ ]:
#@title ▶ Run: connect the MCP socket so the agent uses MCP tools and local tools together
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
import agent

# Socket config: use stdio to start a "homework repo" program next door; the tools live in that program
mcp_client = MultiServerMCPClient({
    "homework_repo": {"command": sys.executable, "args": [str(BASE / "deploy" / "homework_mcp.py")], "transport": "stdio"},
})
mcp_tools = await mcp_client.get_tools()          # In Colab you can write await directly
mcp_names = {t.name for t in mcp_tools}

print("Tools the MCP server can provide:", ", ".join(sorted(mcp_names)))

# Least privilege: this agent only needs to "read" homework, so give it just the two read tools
read_tools = [t for t in mcp_tools if t.name in {"list_homework", "get_homework"}]
available_tools = agent.helper_tools + read_tools
print(f"\n🧰 Tools actually handed to the agent ({len(available_tools)} in total):")
for t in available_tools:
    source = "MCP socket (lives in another program)" if t.name in mcp_names else "local function (lives in this program)"
    print(f"  - {t.name:<14} <- {source}")
    print(f"      description: {' '.join(t.description.split())[:70]}")
    print(f"      parameters: {list(t.args)}")

helper = create_agent(
    model, tools=available_tools,
    system_prompt=agent.base_system_prompt + " The homework students turned in is in the homework repo; use get_homework to get the code.",
)
question = "Look at the homework Chloe turned in and run it to see if there are errors."
print(f"\nUSER: {question}\n")
sequence, step = [], 0
# MCP tools can only be called asynchronously, so this uses astream (not the stream used earlier in 1-B)
async for event in helper.astream({"messages": [{"role": "user", "content": question}]}, stream_mode="updates"):
    for node, update in event.items():
        step += 1
        for msg in (update or {}).get("messages", []):
            if getattr(msg, "tool_calls", None):
                for c in msg.tool_calls:
                    sequence.append(c["name"])
                    source = "MCP socket" if c["name"] in mcp_names else "local function"
                    print(f"--- Step {step} | node: {node} ---  model decided to call -> {c['name']}({c['args']})  <- {source}")
            elif msg.type == "tool":
                print(f"--- Step {step} | node: {node} ---  [result of {msg.name}] {msg.text[:200]}")
            else:
                print(f"--- Step {step} | node: {node} ---  [AIMessage] {msg.text[:300]}")
print("\nOrder of tool use this time:", " -> ".join(sequence) or "(no tools used)")
record("1-B Part 6: connect the MCP socket", {"MCP tools the server provides": sorted(mcp_names),
                                    "tools handed to the agent": [t.name for t in available_tools], "order of tool use": sequence})

### 👀 What did you see?

- The output first lists the **three MCP tools the server can provide**, then the **five tools actually handed to the agent**: the three local tools plus two MCP read-only tools. The basic interface of the tools is the same: each has a name, a description and parameters.
- The difference is **where the tool lives**: the place where `get_homework` really runs is the "homework repo" program next door, not this notebook.
  Other MCP servers can be local or remote; how you authenticate depends on the service, and MCP does not get permissions for you automatically.
- Check the actual record: did it first use the MCP `get_homework` to get the homework, and then the local `run_code` to run it? This example lets you observe **tools from two sources mixed in the same loop**.
- The homework-repo MCP server has `send_email`, but we **deliberately did not hand it to this agent**: reading homework doesn't need sending email, and sent email can't be taken back. Giving only the tools and permissions needed to finish the task is called least privilege (2-C in Session 2 covers it again).
- Here we connect over stdio: the homework repo runs on this same Colab machine. A remote MCP can connect through a network endpoint, and Exercise 1 in 2-E turns the same server into a URL and connects it again.

### 🤔 Prediction questions (usage comparison, optional reading)
About how many tokens do the descriptions of the three local tools take? And the 2 MCP read-only tools actually handed to the agent? If we swap in the teaching list below, which imitates GitHub's tool scale, how much would the estimated length differ?

In [ ]:
#@title ▶ Run: local tools vs. MCP socket, how many tokens the descriptions take
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_mcp_adapters.client import MultiServerMCPClient
import agent

mcp_client = MultiServerMCPClient({
    "homework_repo": {"command": sys.executable, "args": [str(BASE / "deploy" / "homework_mcp.py")], "transport": "stdio"},
})
mcp_tools = await mcp_client.get_tools()
# Count only the two read-only tools really handed to the agent; send_email was not handed over, so it takes no input
read_tools = [t for t in mcp_tools if t.name in {"list_homework", "get_homework"}]

# Illustration only: a tool list at the scale of the GitHub MCP server (names and descriptions are rewritten for teaching, not the official text)
github_scale_tools = [
    ('get_me', 'Get details of the authenticated GitHub user.', {}),
    ('search_repositories', 'Search GitHub repositories by keyword, language, stars or topic.', {'query': 'string', 'page': 'integer', 'per_page': 'integer'}),
    ('get_repository', 'Get metadata of a repository such as description, default branch and visibility.', {'owner': 'string', 'repo': 'string'}),
    ('create_repository', 'Create a new repository for the authenticated user or an organization.', {'name': 'string', 'description': 'string', 'private': 'boolean'}),
    ('fork_repository', 'Fork a repository to your account or an organization.', {'owner': 'string', 'repo': 'string', 'organization': 'string'}),
    ('list_branches', 'List branches in a repository with pagination.', {'owner': 'string', 'repo': 'string', 'page': 'integer'}),
    ('create_branch', 'Create a new branch from an existing branch or commit.', {'owner': 'string', 'repo': 'string', 'branch': 'string', 'from_branch': 'string'}),
    ('list_commits', 'List commits of a branch, optionally filtered by author or path.', {'owner': 'string', 'repo': 'string', 'sha': 'string', 'author': 'string'}),
    ('get_commit', 'Get a single commit including the diff and changed files.', {'owner': 'string', 'repo': 'string', 'sha': 'string'}),
    ('get_file_contents', 'Get the contents of a file or directory from a repository at a given ref.', {'owner': 'string', 'repo': 'string', 'path': 'string', 'ref': 'string'}),
    ('create_or_update_file', 'Create or update a single file in a repository with a commit message.', {'owner': 'string', 'repo': 'string', 'path': 'string', 'content': 'string', 'message': 'string', 'branch': 'string', 'sha': 'string'}),
    ('delete_file', 'Delete a file from a repository with a commit message.', {'owner': 'string', 'repo': 'string', 'path': 'string', 'message': 'string', 'branch': 'string'}),
    ('push_files', 'Push multiple files to a branch in a single commit.', {'owner': 'string', 'repo': 'string', 'branch': 'string', 'files': 'array', 'message': 'string'}),
    ('search_code', 'Search code across GitHub repositories using code search syntax.', {'query': 'string', 'page': 'integer', 'per_page': 'integer'}),
    ('list_issues', 'List issues in a repository filtered by state, labels, assignee or date.', {'owner': 'string', 'repo': 'string', 'state': 'string', 'labels': 'array', 'since': 'string'}),
    ('get_issue', 'Get the details of a specific issue including body and labels.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer'}),
    ('create_issue', 'Create a new issue with title, body, labels and assignees.', {'owner': 'string', 'repo': 'string', 'title': 'string', 'body': 'string', 'labels': 'array', 'assignees': 'array'}),
    ('update_issue', 'Update the title, body, state, labels or assignees of an issue.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer', 'title': 'string', 'body': 'string', 'state': 'string'}),
    ('add_issue_comment', 'Add a comment to an issue or pull request.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer', 'body': 'string'}),
    ('list_issue_comments', 'List comments on an issue with pagination.', {'owner': 'string', 'repo': 'string', 'issue_number': 'integer', 'page': 'integer'}),
    ('search_issues', 'Search issues and pull requests across repositories.', {'query': 'string', 'sort': 'string', 'order': 'string'}),
    ('list_pull_requests', 'List pull requests filtered by state, base or head branch.', {'owner': 'string', 'repo': 'string', 'state': 'string', 'base': 'string', 'head': 'string'}),
    ('get_pull_request', 'Get details of a pull request including mergeability and reviewers.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer'}),
    ('create_pull_request', 'Open a new pull request from a head branch into a base branch.', {'owner': 'string', 'repo': 'string', 'title': 'string', 'head': 'string', 'base': 'string', 'body': 'string', 'draft': 'boolean'}),
    ('update_pull_request', 'Update the title, body, base branch or state of a pull request.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'title': 'string', 'body': 'string'}),
    ('merge_pull_request', 'Merge a pull request using merge, squash or rebase.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'merge_method': 'string'}),
    ('get_pull_request_files', 'List files changed in a pull request with patch snippets.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer'}),
    ('get_pull_request_diff', 'Get the unified diff of a pull request.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer'}),
    ('create_pull_request_review', 'Create a review on a pull request with comments and a verdict.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'body': 'string', 'event': 'string', 'comments': 'array'}),
    ('request_reviewers', 'Request reviewers for a pull request.', {'owner': 'string', 'repo': 'string', 'pull_number': 'integer', 'reviewers': 'array'}),
    ('list_releases', 'List releases of a repository.', {'owner': 'string', 'repo': 'string', 'page': 'integer'}),
    ('get_latest_release', 'Get the latest published release of a repository.', {'owner': 'string', 'repo': 'string'}),
    ('list_tags', 'List git tags of a repository.', {'owner': 'string', 'repo': 'string'}),
    ('list_workflows', 'List GitHub Actions workflows defined in a repository.', {'owner': 'string', 'repo': 'string'}),
    ('run_workflow', 'Trigger a workflow_dispatch run of a GitHub Actions workflow.', {'owner': 'string', 'repo': 'string', 'workflow_id': 'string', 'ref': 'string', 'inputs': 'object'}),
    ('get_workflow_run', 'Get the status and conclusion of a workflow run.', {'owner': 'string', 'repo': 'string', 'run_id': 'integer'}),
    ('get_job_logs', 'Download logs of jobs in a workflow run, optionally only failed jobs.', {'owner': 'string', 'repo': 'string', 'run_id': 'integer', 'failed_only': 'boolean'}),
    ('list_notifications', 'List notifications for the authenticated user.', {'all': 'boolean', 'participating': 'boolean', 'since': 'string'}),
    ('list_code_scanning_alerts', 'List code scanning alerts of a repository by state and severity.', {'owner': 'string', 'repo': 'string', 'state': 'string', 'severity': 'string'}),
    ('list_secret_scanning_alerts', 'List secret scanning alerts detected in a repository.', {'owner': 'string', 'repo': 'string', 'state': 'string'}),
]

def to_tool_spec(name, desc, params):
    props = {k: {"type": v, "description": k} for k, v in params.items()}
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": list(params)[:2]}}}

def estimate_tool_spec(specs):
    return estimate_tokens(json.dumps(specs, ensure_ascii=False))

groups = [
    ("Local tools (our own)", [convert_to_openai_tool(t) for t in agent.helper_tools]),
    ("Homework repo MCP (handed to agent)", [convert_to_openai_tool(t) for t in read_tools]),
    ("GitHub-scale MCP (illustration)", [to_tool_spec(*t) for t in github_scale_tools]),
]
results = {}
for name, specs in groups:
    results[name] = estimate_tool_spec(specs)
    print(f"{name:<38} {len(specs):>3} tools: about {results[name]:>5,} tokens  " + "█" * max(1, results[name] // 100))
local = results["Local tools (our own)"]
large = results["GitHub-scale MCP (illustration)"]
print(f"\nIf you handed this whole GitHub-scale list to the model, one call would carry about {large:,} tokens of tool descriptions, {large / local:.1f} times the local tools.")
print("(The token counts are estimated from text length; only the relative sizes matter.)")
record("1-B Part 6: tokens of MCP tool descriptions", results)

### 👀 What did you see?

- The homework-repo server has 3 tools, but only 2 were handed to the agent, so only those 2 descriptions are counted; this illustration list has 40, and its descriptions are clearly longer. The real cost depends on which tools you choose, the model's format and its tokenizer, so here we only look at the relative sizes of the estimates.
- To an agent, local tools and MCP tools are used the same way; the differences are where the tool lives, how it is authenticated, and how much of the desk its descriptions take up.
- Session 2 uses **Skills** to load task guidance on demand. A Skill does not automatically remove tool definitions; to shorten the tool list, your program still has to filter it or provide tools later.

---
## Part 7 (optional after class): Read the agent's core loop with LangGraph

> 📍 **Matching slides**: slide 47 "Optional appendix: the core model-and-tool loop"

The core exercises already used `create_agent`. This part instead uses LangGraph to demonstrate the "model <-> tool" loop; it is not a full implementation of `create_agent`, and you can read it after class.

It reuses the three components from 1-A: **State** holds the messages, **Node** does the work, **Edge** connects to the next step; a conditional edge picks the route based on tool requests:
a node that asks the model, `ask_model`; a node that runs the tools, `tools`; and an if for "is a tool wanted?" (`tools_condition`), looped into a circle.
The code is `make_expanded_agent` in `agent.py`.

| Component | What it does |
| --- | --- |
| `@tool` | Turns a function into a tool the model can see |
| `bind_tools` | Attaches the tool descriptions to every call |
| `ask_model` | Asks the model; it returns text or tool_calls |
| `ToolNode` | Reads tool_calls, runs them, and wraps the results |
| `tools_condition` | Built-in routing: with tool_calls -> tools, without -> END |
| `add_edge("tools", "ask_model")` | After the tool runs, go back and ask the model, forming the cycle |

### 🤔 Prediction questions
How many nodes will the flow diagram have? Which edge is the conditional edge?

In [ ]:
#@title ▶ Run: build the core loop yourself with LangGraph
homework = "Variable typo"  #@param ["Variable typo", "Correct program", "Messy indentation", "Missing parenthesis", "My homework"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import agent

expanded = agent.make_expanded_agent(model)
show_graph(expanded)
events = agent.pretty_trace(expanded, agent.homework_message(load_homework(homework)))
record(f"1-B Part 7: expanded agent ({homework})", events)

### 👀 What did you see?

- Not counting START/END, there are two working nodes: `ask_model` and `tools`. The edge out of `ask_model` is the conditional edge (dashed): if a tool is wanted, go to `tools`; if not, end.
- After `tools` finishes it always returns to `ask_model`, and that backward-pointing arrow is the loop.
- This session builds agents with LangChain's `create_agent`; what it runs on underneath is LangGraph.

---
## Wrapping up Session 1

- **Agent = model + tools + loop**; the model only "says" which tool it wants, and your program does the work.
- **The flow arranges the steps that cannot be skipped, and the agent handles the parts that need flexible judgment**; the two can be nested (a fixed flow outside, with an agent inside one station). That is "Combining workflow and agent: an agent inside a node" on slide 42, and it is what "dual-core" in the course name means (you will build it in 2-D in Session 2).
- 📝 **Homework after class**: put a homework you wrote yourself into "My homework", run it once in 1-A and once in 1-B,
  take a screenshot of the State history from 1-A and the tool-call record from 1-B, and write one sentence on which result you trust more and why.
  When you use "My homework", first change the "spec" in Part 3 to the assignment for your own homework; Parts 4 and 7 reuse the same spec, and rerunning "0. Environment setup" resets it to the default.

## Wrap-up: save your work and hand it in

- 💾 **Save to Google Drive**: save a copy of `/content/review-bot/` (your edited handbooks, code, and State records) to your own Google Drive.
- ⬇️ **Download the zip**: download it straight to your computer.
- 📸 **The last cell**: prints the results the experiment cells saved. Pick the relevant records for a screenshot, then add your observations and reasons as the assignment asks.

These are the results the experiment cells saved through `record()`, not every intermediate state of every run.
Records in the same runtime pile up, so they may include experiments from other notebooks; use the printed titles to tell them apart.

A temporary browser disconnect does not necessarily wipe your files; files under `/content/` are lost only when the runtime is deleted or recycled.
To keep your handbooks, code, and checkpoint, back them up first. The notebook itself is saved with the menu "File → Save a copy in Drive". The two kinds of saving do different jobs.
When you come back, first check that the runtime is still there. If it was reset, re-run "0. Setup" and this part's setup cell, then restore the files you backed up as needed.

In [ ]:
#@title 💾 Save to Google Drive (an authorization window will pop up)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
save_to_drive()

In [ ]:
#@title ⬇️ Download the zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
download_zip()

In [ ]:
#@title 📸 Print the recorded experiment results (pick the relevant records to hand in)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
print_history()